# Errors and Debugging

The most common type of errors at runtime stems from worker tasks.
A first measure to preventing this is using the correct types, but still something might go wrong.
In this example we will produce an error and investigate it with the debugging tools in tierkreis.

## Worker Errors

Worker errors can occur in multiple ways.
For python workers an error occurs when an uncaught exception raises.
For other workers (including python) a non-zero exit code will also produce an error.

Defining a graph that will always run an error:

In [ ]:
from error_worker import fail
from tierkreis.builder import Graph, Workflow
from tierkreis.controller.data.core import EmptyModel
from tierkreis.controller.data.models import TKR


def error_graph() -> Workflow:
    g = Graph(EmptyModel, TKR[str])
    output = g.task(fail())
    return g.finish_with_outputs(output)

The task `fail` will make `wait_for` raise a `ValueError` containing the worker diagnostic (`"I refuse!"`):

In [ ]:
from tierkreis.runtime import new_default

runtime = await new_default()
with runtime:
    workflow_id = await runtime.save_workflow("error handling", error_graph())
    run_id = await runtime.start_new_run(workflow_id, {})
    try:
        await runtime.wait_for(run_id)
    except ValueError as error:
        print(error)

## Debugging

The run remains inspectable after failure. `get_summary` identifies failed locations, and `debug_read_node_states` exposes their runtime state and execution context. Enable runtime debug logs with `RuntimeConfig.set_log_level("debug")` or `tkr run -v`.

In [ ]:
summary = await runtime.get_summary(run_id)
print(summary.errored_locations)
states = await runtime.debug_read_node_states(run_id, 0, summary.errored_locations)
print(states)

The Rust runtime does not currently implement legacy Python breakpoints. Instead, restart all failed nodes by omitting the location list. This creates a new attempt and preserves attempt zero. This worker always fails, so the restarted attempt will also report an error.

In [ ]:
with runtime:
    new_attempt, invalidated = await runtime.restart_task(run_id)
    try:
        await runtime.wait_for(run_id, new_attempt)
    except ValueError:
        pass

print(new_attempt, invalidated)